# Adzuna Scraping

In [ ]:
import requests
import pandas as pd
import time

APP_ID, APP_KEY = "YOUR_APP_ID", "YOUR_APP_KEY"
BASE = "https://api.adzuna.com/v1/api/jobs/it/search/{page}"

def fetch_adzuna(query="data scientist", max_pages=10):
    rows = []
    for page in range(1, max_pages + 1):
        r = requests.get(
            BASE.format(page=page),
            params={
                "app_id": APP_ID,
                "app_key": APP_KEY,
                "what": query,
                "results_per_page": 50,
                "content-type": "application/json",
            },
            timeout=30,
        )
        r.raise_for_status()
        results = r.json().get("results", [])
        if not results:
            break
        for j in results:
            rows.append({
                "source": "adzuna",
                "title": j.get("title"),
                "company": (j.get("company") or {}).get("display_name"),
                "location": (j.get("location") or {}).get("display_name"),
                "salary_min": j.get("salary_min"),
                "salary_max": j.get("salary_max"),
                "salary_is_predicted": j.get("salary_is_predicted"),
                "created": j.get("created"),
                "url": j.get("redirect_url"),
            })
        time.sleep(0.5)  # be polite with rate limits
    return pd.DataFrame(rows)

df = fetch_adzuna()
df.to_csv("data_scientist_italy_adzuna.csv", index=False)
print(df.head(), len(df))